# Topic 01. 확률분포와 중심극한정리

데이터관리론의 확률 기초를 실행 가능한 예제로 정리합니다.
주사위 합의 PMF/CDF, 베르누이·이항분포, 중심극한정리를 순서대로 확인합니다.


## 제출자 정보

- 이름: 이성민
- 학과: 소프트웨어융합과
- 학년: 2학년
- 학번: 2151050


## 1. 이산 확률변수: 두 주사위 합의 PMF와 CDF

PMF는 각 값이 나올 확률이고, CDF는 특정 값 이하가 나올 누적확률입니다.
두 주사위를 모두 나열하면 표본공간이 작아서 정확한 확률을 직접 계산할 수 있습니다.


In [1]:
from itertools import product

import pandas as pd

outcomes = [a + b for a, b in product(range(1, 7), repeat=2)]
pmf = (
    pd.Series(outcomes, name="sum")
    .value_counts(normalize=True)
    .sort_index()
    .rename("pmf")
    .to_frame()
)
pmf["cdf"] = pmf["pmf"].cumsum()
pmf


,pmf,cdf
sum,,
2,0.027778,0.027778
3,0.055556,0.083333
4,0.083333,0.166667
5,0.111111,0.277778
6,0.138889,0.416667
7,0.166667,0.583333
8,0.138889,0.722222
9,0.111111,0.833333
10,0.083333,0.916667


## 2. 베르누이와 이항분포 시뮬레이션

베르누이 시행은 성공/실패가 있는 단일 시행입니다.
같은 시행을 여러 번 반복해 성공 횟수를 세면 이항분포가 됩니다.


In [2]:
import numpy as np
from scipy.stats import binom

rng = np.random.default_rng(2151050)
flips_per_trial = 100
trials = 10_000
simulated_success = rng.binomial(n=flips_per_trial, p=0.5, size=trials)

summary = {
    "simulation_mean": round(float(simulated_success.mean()), 2),
    "theoretical_mean": flips_per_trial * 0.5,
    "simulation_std": round(float(simulated_success.std(ddof=1)), 2),
    "theoretical_std": round(float(binom.std(flips_per_trial, 0.5)), 2),
}
summary


{'simulation_mean': 49.98,
 'theoretical_mean': 50.0,
 'simulation_std': 5.02,
 'theoretical_std': 5.0}

## 3. 중심극한정리 확인

원래 데이터가 균등분포처럼 생겼더라도 표본 평균을 반복해서 모으면 평균의 분포는 점점 정규분포에 가까워집니다.
아래 코드는 표본 크기가 커질수록 표본평균의 표준편차가 작아지는 것을 보여줍니다.


In [3]:
import matplotlib.pyplot as plt
import pandas as pd

population = rng.integers(1, 7, size=100_000)
records = []

for sample_size in [1, 5, 30]:
    sample_means = [
        rng.choice(population, size=sample_size, replace=True).mean()
        for _ in range(2_000)
    ]
    records.append(
        {
            "sample_size": sample_size,
            "mean_of_sample_means": round(float(np.mean(sample_means)), 3),
            "std_of_sample_means": round(float(np.std(sample_means, ddof=1)), 3),
        }
    )

clt_summary = pd.DataFrame(records)

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(clt_summary["sample_size"], clt_summary["std_of_sample_means"], marker="o")
ax.set_title("Sample Size vs Std of Sample Means")
ax.set_xlabel("Sample Size")
ax.set_ylabel("Std")
plt.close(fig)

clt_summary


,sample_size,mean_of_sample_means,std_of_sample_means
0,1,3.478,1.696
1,5,3.476,0.771
2,30,3.499,0.312


## 정리

- PMF는 개별 확률, CDF는 누적 확률입니다.
- 베르누이 시행을 여러 번 반복하면 이항분포가 됩니다.
- 중심극한정리는 표본평균 기반 추론의 핵심 근거입니다.
